In [20]:
import os
import sys
import numpy as np
import pandas as pd
import warnings
import importlib

pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")

# 프로젝트 루트 디렉토리 설정 (노트북 위치를 기준으로 상위 디렉토리로 이동)
# 노트북 위치: 03_배포용/01_train/01_segmentation/1_segmentation_kmodes.ipynb
# 프로젝트 루트: 01_예측/
current_dir = os.getcwd()
# 현재 디렉토리에서 프로젝트 루트 찾기 (03_배포용 폴더가 있는 위치)
if '03_배포용' in current_dir:
    # 03_배포용이 경로에 포함되어 있으면 그 위가 프로젝트 루트
    project_root = current_dir.split('03_배포용')[0]
else:
    # 그렇지 않으면 현재 디렉토리에서 상위로 이동하여 찾기
    project_root = current_dir
    while not os.path.exists(os.path.join(project_root, '03_배포용')):
        parent = os.path.dirname(project_root)
        if parent == project_root:  # 루트 디렉토리에 도달
            project_root = os.getcwd()  # 기본값으로 현재 디렉토리 사용
            break
        project_root = parent

# 프로젝트 루트로 작업 디렉토리 변경
os.chdir(project_root)

# 현재 노트북의 디렉토리를 sys.path에 추가 (숫자로 시작하는 모듈명 import를 위해)
notebook_dir = os.path.join(project_root, '03_배포용/01_train/01_segmentation')
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

# prepare_for_seg 모듈에서 전처리 함수들 import
# 파일명이 01_prepare_for_seg.py이므로 importlib.import_module 사용
prepare_for_seg = importlib.import_module('01_prepare_for_seg')
importlib.reload(prepare_for_seg)

# 모듈에서 함수들을 직접 가져오기
apply_optimal_dtypes = prepare_for_seg.apply_optimal_dtypes
add_generation_features_optimal = prepare_for_seg.add_generation_features_optimal
create_price_binned_feature = prepare_for_seg.create_price_binned_feature
create_engine_binned_feature = prepare_for_seg.create_engine_binned_feature
remove_outliers_by_new_car_price = prepare_for_seg.remove_outliers_by_new_car_price
remove_outliers_by_band = prepare_for_seg.remove_outliers_by_band
add_국산외제차_구분 = prepare_for_seg.add_국산외제차_구분
assign_interest_rate_by_sale_date = prepare_for_seg.assign_interest_rate_by_sale_date
add_accident_rank_full = prepare_for_seg.add_accident_rank_full
add_car_age_features = prepare_for_seg.add_car_age_features
add_mileage_features = prepare_for_seg.add_mileage_features
add_temporal_features_normalized = prepare_for_seg.add_temporal_features_normalized

# ============================================================
# 1. 데이터 로드
# ============================================================
print("데이터 로드 중...")
file_path = os.path.join(project_root, "03_배포용/00_data/02_combined_data_gen/df_car_combined_gen.csv")
print(f"파일 경로: {file_path}")
print(f"파일 존재 여부: {os.path.exists(file_path)}")
df_car = pd.read_csv(file_path, low_memory=False)
print(f"✓ 데이터 로드 완료: {len(df_car):,}건")

# ============================================================
# 2. 데이터 타입 최적화 (apply_optimal_dtypes)
# 먼저 실행: 숫자 정리, 타입 변환, 신차가격 만원 단위 변환 포함
# ============================================================
print("\n데이터 타입 최적화 중...")
df_car = apply_optimal_dtypes(df_car)
print("✓ 데이터 타입 최적화 완료")
print(f"  - 신차가격: 만원 단위로 변환됨")
print(f"  - 숫자형 컬럼: 최적 타입으로 변환됨")
print(f"  - 날짜형 컬럼: datetime으로 변환됨")
print(f"  - 범주형 컬럼: category로 변환됨")
print(f"  - 연월식: {df_car['연월식'].dtype} 타입으로 변환됨 (Period 타입이어야 함)")

# ============================================================
# 3. 세대 관련 피처 추가
# ============================================================
print("\n세대 관련 피처 추가 중...")
df_car = add_generation_features_optimal(df_car)
print("✓ 세대 관련 피처 추가 완료")

# ============================================================
# 4. 배기량 구간 피처 생성
# ============================================================
print("\n배기량 구간 피처 생성 중...")
df_car, engine_dist = create_engine_binned_feature(
    df_car,
    disp_col="배기량",
    new_col_name="배기량_구간",
)
print("✓ 배기량 구간 피처 생성 완료")
print(engine_dist)

# ============================================================
# 5. 신차가격 구간 피처 생성
# ============================================================
print("\n신차가격 구간 피처 생성 중...")
df_car, price_dist = create_price_binned_feature(
    df_car,
    col="신차가격(선택옵션포함)",
    new_col_name="신차가격_구간",
)
print("✓ 신차가격 구간 피처 생성 완료")
print(price_dist)

# ============================================================
# 5-1. 국산차/외제차 구분 피처 생성
# ============================================================
print("\n국산차/외제차 구분 피처 생성 중...")
# 제조사코드를 기준으로 구분 (제조사코드가 이미 정확하므로 normalize 불필요)
# 국산차=0, 외제차=1
df_car = add_국산외제차_구분(
    df_car, 
    제조사코드_col='제조사코드',
    new_col_name='국산외제차'
)
print("✓ 국산차/외제차 구분 피처 생성 완료")
print("  - 국산차: 0 (제조사코드: 1, 2, 3, 4, 5, 7)")
print("  - 외제차: 1 (나머지 제조사코드)")
print("\n분포:")
print(df_car['국산외제차'].value_counts().sort_index())
print(f"\n국산차 비율: {(df_car['국산외제차'] == 0).sum() / len(df_car) * 100:.2f}%")
print(f"외제차 비율: {(df_car['국산외제차'] == 1).sum() / len(df_car) * 100:.2f}%")

# ============================================================
# 6. 이상치 제거
# ============================================================
print("\n이상치 제거 중...")

# 6-1) 신차가격 대비 판매신고가 이상치 제거
print("\n[1단계] 신차가격 대비 판매신고가 이상치 제거")
df_car = remove_outliers_by_new_car_price(
    df_car,
    new_car_price_col="신차가격(선택옵션포함)",
    sale_price_col="판매신고가"
)

# 6-2) 광고가 대비 판매신고가 이상치 제거
print("\n[2단계] 광고가 대비 판매신고가 이상치 제거")
df_car = remove_outliers_by_band(
    df_car,
    ad_price_col="최초 광고가",
    sale_price_col="판매신고가",
    threshold=0.40
)

# ============================================================
# 6-3. 추가 피처 생성
# ============================================================
print("\n추가 피처 생성 중...")

# 판매기준금리 매핑
df_car = assign_interest_rate_by_sale_date(
    df_car,
    sale_col="판매신고일",
    new_col="판매기준금리"
)
print("✓ 판매기준금리 매핑 완료")

# 사고 심각도 랭크 생성
df_car = add_accident_rank_full(
    df_car,
    col_acc_flag="사고유무",
    col_replace="교환부위",
    col_panel="판금부위",
    new_col_severity="사고_심각도"
)
print("✓ 사고_심각도 피처 생성 완료")

# 차량 나이 피처 생성
df_car = add_car_age_features(
    df_car,
    prod_ym_col="연월식",
    sale_date_col="판매신고일",
    out_year_col="차나이_년",
    out_month_col="차나이_월",
    max_year_cap=20
)
print("✓ 차나이_년, 차나이_월 피처 생성 완료")

# 주행거리 관련 피처 생성
df_car = add_mileage_features(
    df_car,
    mileage_col="주행거리",
    prod_ym_col="연월식",
    sale_date_col="판매신고일",
    month_smoothing=1,
    avg_km_per_year=20_000
)
print("✓ 주행거리 관련 피처 생성 완료 (km_per_month, log_km_per_month, mileage_ratio)")

# 시간 관련 피처 생성
df_car = add_temporal_features_normalized(
    df_car,
    sale_date_col="판매신고일"
)
print("✓ 시간 관련 피처 생성 완료 (sale_month, sale_week, sale_day, sin/cos 피처)")

print("\n✓ 전처리 완료!")
print(f"최종 데이터 행 수: {len(df_car):,}건")
print(f"최종 데이터 컬럼 수: {len(df_car.columns)}개")



데이터 로드 중...
파일 경로: /Users/dan/Desktop/코오롱_프로젝트/01_예측/03_배포용/00_data/02_combined_data_gen/df_car_combined_gen.csv
파일 존재 여부: True
✓ 데이터 로드 완료: 162,573건

데이터 타입 최적화 중...


✓ 데이터 타입 최적화 완료
  - 신차가격: 만원 단위로 변환됨
  - 숫자형 컬럼: 최적 타입으로 변환됨
  - 날짜형 컬럼: datetime으로 변환됨
  - 범주형 컬럼: category로 변환됨
  - 연월식: period[M] 타입으로 변환됨 (Period 타입이어야 함)

세대 관련 피처 추가 중...
✓ 세대 관련 피처 추가 완료

배기량 구간 피처 생성 중...
✓ 배기량 구간 피처 생성 완료
             count    pct
배기량_구간                   
0cc           5663   3.48
500~999cc    11078   6.81
1000~1499cc   1920   1.18
1500~1999cc  68281  42.00
2000~2499cc  42184  25.95
2500~2999cc  14566   8.96
3000cc~      18881  11.61

신차가격 구간 피처 생성 중...
✓ 신차가격 구간 피처 생성 완료
                   count    pct
신차가격_구간                        
2천만원 미만            17261  10.62
2천만원 이상 ~ 3천만원 미만  29346  18.05
3천만원 이상 ~ 4천만원 미만  38265  23.54
4천만원 이상 ~ 5천만원 미만  24863  15.29
5천만원 이상 ~ 7천만원 미만  27404  16.86
7천만원 이상 ~ 1억원 미만   15808   9.72
1억원 이상              9626   5.92

국산차/외제차 구분 피처 생성 중...
✓ 국산차/외제차 구분 피처 생성 완료
  - 국산차: 0 (제조사코드: 1, 2, 3, 4, 5, 7)
  - 외제차: 1 (나머지 제조사코드)

분포:
국산외제차
0    127211
1     35362
Name: count, dtype: int64

국산차 비율: 78.25%
외제차 비율: 21.75%

이상치 제거 중..

In [21]:
# ============================================================
# 7. Train/Test 분리 (판매신고일 기준: 마지막 달을 Test, 나머지를 Train)
# ============================================================
print("\nTrain/Test 분리 중...")

# 판매신고일의 연월(YYYY-MM) 추출
df_car['판매신고일_연월'] = df_car['판매신고일'].dt.to_period('M')

# 고유한 연월 목록 추출 및 정렬
unique_months = sorted(df_car['판매신고일_연월'].dropna().unique())

print(f"전체 고유 연월 수: {len(unique_months)}개")
print(f"고유 연월 목록: {unique_months}")

# 마지막 달을 자동으로 찾아서 Test로 설정, 나머지는 Train
if len(unique_months) > 0:
    last_month = unique_months[-1]  # 마지막 달
    train_months = unique_months[:-1]  # 마지막 달을 제외한 모든 달
    test_months = [last_month]  # 마지막 달만 Test
    
    print(f"마지막 달 (Test): {last_month}")
    if len(train_months) > 0:
        print(f"Train 기간: {train_months[0]} ~ {train_months[-1]} ({len(train_months)}개월)")
    else:
        print(f"Train 기간: 없음 (0개월)")
    print(f"Test 기간: {test_months[0]} ~ {test_months[-1]} ({len(test_months)}개월)")
else:
    print("⚠ 경고: 데이터에 유효한 연월이 없습니다.")
    train_months = []
    test_months = []

# Train/Test 분할
train_mask = df_car['판매신고일_연월'].isin(train_months)
test_mask = df_car['판매신고일_연월'].isin(test_months)

df_train = df_car[train_mask].copy()
df_test = df_car[test_mask].copy()

# 판매신고일_연월 컬럼 제거 (일시적으로만 사용)
df_train = df_train.drop(columns=['판매신고일_연월'], axis=1, errors='ignore')
df_test = df_test.drop(columns=['판매신고일_연월'], axis=1, errors='ignore')

print(f"\nTrain set size: {len(df_train):,}건")
print(f"Test set size: {len(df_test):,}건")
print("✓ Train/Test 분리 완료")



Train/Test 분리 중...
전체 고유 연월 수: 11개
고유 연월 목록: [Period('2025-01', 'M'), Period('2025-02', 'M'), Period('2025-03', 'M'), Period('2025-04', 'M'), Period('2025-05', 'M'), Period('2025-06', 'M'), Period('2025-07', 'M'), Period('2025-08', 'M'), Period('2025-09', 'M'), Period('2025-10', 'M'), Period('2025-11', 'M')]
마지막 달 (Test): 2025-11
Train 기간: 2025-01 ~ 2025-10 (10개월)
Test 기간: 2025-11 ~ 2025-11 (1개월)

Train set size: 148,235건
Test set size: 11,440건
✓ Train/Test 분리 완료


In [22]:
# ============================================================
# 8. Train set에서 통계량 계산 (등급코드 + 모델코드 기준)
# ============================================================
# 
# 등급코드 기준만 사용하는 이유:
# 1. 계층 구조: 제조사 → 대표차종명 → 모델코드 → 등급코드 → 세부등급코드
#    - 대표차종명(384개)이 이미 클러스터링에 포함되어 있음
#    - 모델코드(706개)는 대표차종명의 하위 개념이므로 중복될 수 있음
#    - 등급코드(3,697개)는 모델코드의 하위 개념으로 더 세분화된 특성 반영
#
# 2. 등급코드 기준: 더 세분화된 특성 (3,697개 등급코드)
#    - 각 등급코드별로 DOM 평균을 계산하여 판매속도지수 산출
#    - 각 등급코드별로 빈도 계산
#    - 하나의 모델코드에 여러 등급코드가 포함될 수 있고, 각 등급코드의 판매속도가 다를 수 있음
#
# 3. KModes 클러스터링 관점:
#    - 등급코드(3,697개)는 너무 많아서 KModes 클러스터링 컬럼에서 제외
#    - 하지만 등급코드 기준 통계량(판매속도지수, 빈도)을 사용하면 세분화된 특성 반영 가능
#    - 등급코드 기준 통계량을 구간화하여 KModes에 사용
#    - 카테고리 수 감소: 등급코드 3,697개 → 구간 5개
#    - 대표차종명이 이미 모델코드의 상위 개념을 포함하므로 모델코드 기준 통계량은 불필요
#
# ============================================================

# prepare_for_seg 모듈에서 함수들 가져오기 (이미 첫 번째 셀에서 로드됨)
# 숫자로 시작하는 모듈명이므로 importlib 사용
import importlib
prepare_for_seg = importlib.import_module('01_prepare_for_seg')

# 함수들을 변수에 할당
calculate_판매속도지수_stats = prepare_for_seg.calculate_판매속도지수_stats
apply_판매속도지수 = prepare_for_seg.apply_판매속도지수
calculate_빈도_stats = prepare_for_seg.calculate_빈도_stats
apply_빈도 = prepare_for_seg.apply_빈도

print("\n통계량 계산 중...")
print("  기준: 등급코드 (세분화된 특성 반영)")
print("  참고: 대표차종명이 이미 클러스터링에 포함되어 있어 모델코드 기준 통계량은 제외")

# Train set에서 등급코드별 판매속도지수 계산
# 각 등급코드별 DOM 평균을 계산하여 판매속도지수 산출
판매속도지수_등급코드_stats = calculate_판매속도지수_stats(
    df_train, 
    group_col='등급코드',  # 등급코드 기준
    ad_date_col='최초 광고 등록일',
    sale_date_col='판매신고일',
    multiplier=100
)
print(f"✓ 등급코드별 판매속도지수 계산 완료 ({len(판매속도지수_등급코드_stats)}개 등급코드)")
print(f"  - 유효 판매속도지수: {판매속도지수_등급코드_stats.notna().sum():,}개")
print(f"  - 평균 판매속도지수: {판매속도지수_등급코드_stats.mean():.2f}")

# Train set에서 등급코드별 빈도 계산
# 각 등급코드가 Train set에서 몇 번 나타나는지 계산
빈도_등급코드_stats = calculate_빈도_stats(df_train, group_col='등급코드')  # 등급코드 기준
print(f"✓ 등급코드별 빈도 계산 완료 ({len(빈도_등급코드_stats)}개 등급코드)")
print(f"  - 평균 빈도: {빈도_등급코드_stats.mean():.1f}건")
print(f"  - 중앙값 빈도: {빈도_등급코드_stats.median():.0f}건")



print("✓ 통계량 계산 완료")



통계량 계산 중...
  기준: 등급코드 (세분화된 특성 반영)
  참고: 대표차종명이 이미 클러스터링에 포함되어 있어 모델코드 기준 통계량은 제외
✓ 등급코드별 판매속도지수 계산 완료 (2616개 등급코드)
  - 유효 판매속도지수: 2,616개
  - 평균 판매속도지수: 1.64
✓ 등급코드별 빈도 계산 완료 (2616개 등급코드)
  - 평균 빈도: 56.7건
  - 중앙값 빈도: 11건
✓ 통계량 계산 완료


In [23]:
# ============================================================
# 9. Train/Test에 통계량 매핑 (등급코드 기준)
# ============================================================
# Train set에서 계산한 등급코드별 통계량을 Train/Test set에 매핑
# - Train set: Train set 기준으로 계산된 통계량을 매핑 (일관성 유지)
# - Test set: Train set 기준 통계량을 매핑 (데이터 누수 방지)
# ============================================================

print("\n통계량 매핑 중...")
print("  기준: 등급코드 (Train set 기준 통계량 사용)")

# Train set에 등급코드 기준 판매속도지수 매핑
# 컬럼명: '판매속도지수_등급코드'
df_train = apply_판매속도지수(df_train, 판매속도지수_등급코드_stats, group_col='등급코드')
print("✓ Train set에 등급코드 기준 판매속도지수 매핑 완료 (컬럼: 판매속도지수_등급코드)")

# Train set에 등급코드 기준 빈도 매핑 (모델 학습 시 필요)
# 컬럼명: '빈도_등급코드'
df_train = apply_빈도(df_train, 빈도_등급코드_stats, group_col='등급코드')
print("✓ Train set에 등급코드 기준 빈도 매핑 완료 (컬럼: 빈도_등급코드)")

# Test set에 등급코드 기준 판매속도지수 매핑 (Train set 기준 통계량 사용)
# Test set의 등급코드가 Train set에 없으면 NaN이 됨
df_test = apply_판매속도지수(df_test, 판매속도지수_등급코드_stats, group_col='등급코드')
print("✓ Test set에 등급코드 기준 판매속도지수 매핑 완료 (컬럼: 판매속도지수_등급코드)")

# Test set에 등급코드 기준 빈도 매핑 (Train set 기준 통계량 사용)
df_test = apply_빈도(df_test, 빈도_등급코드_stats, group_col='등급코드')
print("✓ Test set에 등급코드 기준 빈도 매핑 완료 (컬럼: 빈도_등급코드)")

print("✓ 통계량 매핑 완료")



통계량 매핑 중...
  기준: 등급코드 (Train set 기준 통계량 사용)
✓ Train set에 등급코드 기준 판매속도지수 매핑 완료 (컬럼: 판매속도지수_등급코드)
✓ Train set에 등급코드 기준 빈도 매핑 완료 (컬럼: 빈도_등급코드)
✓ Test set에 등급코드 기준 판매속도지수 매핑 완료 (컬럼: 판매속도지수_등급코드)
✓ Test set에 등급코드 기준 빈도 매핑 완료 (컬럼: 빈도_등급코드)
✓ 통계량 매핑 완료


In [24]:
# ============================================================
# 9-1. 판매속도지수와 빈도를 구간으로 나누기 (KModes용)
# ============================================================
# KModes는 범주형 데이터만 사용하므로, 연속형인 판매속도지수와 빈도를 구간화 필요
# - Train set 분포 기준으로 구간 경계 결정 (quantile 방법)
# - Test set에는 Train set의 구간 경계를 동일하게 적용 (데이터 누수 방지)
# - 등급코드 기준 통계량을 구간화하여 KModes에 사용
# ============================================================

# prepare_for_seg 모듈에서 함수들 가져오기 (이미 첫 번째 셀에서 로드됨)
# 숫자로 시작하는 모듈명이므로 importlib 사용
import importlib
prepare_for_seg = importlib.import_module('01_prepare_for_seg')

# 함수들을 변수에 할당
create_판매속도지수_binned_feature = prepare_for_seg.create_판매속도지수_binned_feature
create_빈도_binned_feature = prepare_for_seg.create_빈도_binned_feature

print("\n통계량 구간화 중...")
print("  기준: 등급코드 기준 통계량 (판매속도지수_등급코드, 빈도_등급코드)")
print("  방법: K-means 클러스터링 기반 (분포의 자연스러운 경계 찾기)")

# Train set에서 등급코드 기준 판매속도지수 구간화 (Train set 분포 기준)
# 등급코드별 판매속도지수를 5개 구간으로 나눔 (K-means 방법으로 분포의 자연스러운 경계 찾기)
df_train, 판매속도지수_등급코드_dist, 판매속도지수_등급코드_bins = create_판매속도지수_binned_feature(
    df_train,
    col='판매속도지수_등급코드',  # 등급코드 기준 판매속도지수
    new_col_name='판매속도지수_등급코드_구간',
    n_bins=5,
    method='kmeans'  # 분포의 자연스러운 경계를 찾기 위해 K-means 사용
)
print("✓ Train set 등급코드 기준 판매속도지수 구간화 완료")
print(f"  컬럼명: 판매속도지수_등급코드_구간")
print(판매속도지수_등급코드_dist)

# Train set에서 등급코드 기준 빈도 구간화 (Train set 분포 기준)
# 등급코드별 빈도를 5개 구간으로 나눔 (K-means 방법으로 분포의 자연스러운 경계 찾기)
df_train, 빈도_등급코드_dist, 빈도_등급코드_bins = create_빈도_binned_feature(
    df_train,
    col='빈도_등급코드',  # 등급코드 기준 빈도
    new_col_name='빈도_등급코드_구간',
    n_bins=5,
    method='kmeans'  # 분포의 자연스러운 경계를 찾기 위해 K-means 사용
)
print("\n✓ Train set 등급코드 기준 빈도 구간화 완료")
print(f"  컬럼명: 빈도_등급코드_구간")
print(빈도_등급코드_dist)

# Test set에도 동일한 구간 적용 (Train set의 구간 경계 사용)
# 데이터 누수 방지를 위해 Train set에서 결정한 구간 경계를 그대로 사용

# 등급코드 기준 통계량 구간화
df_test, _, _ = create_판매속도지수_binned_feature(
    df_test,
    col='판매속도지수_등급코드',
    new_col_name='판매속도지수_등급코드_구간',
    bins=판매속도지수_등급코드_bins  # Train set의 구간 경계 사용
)

df_test, _, _ = create_빈도_binned_feature(
    df_test,
    col='빈도_등급코드',
    new_col_name='빈도_등급코드_구간',
    bins=빈도_등급코드_bins  # Train set의 구간 경계 사용
)



print("\n✓ Test set 구간화 완료 (Train set 구간 경계 사용)")
print("✓ 통계량 구간화 완료")



통계량 구간화 중...
  기준: 등급코드 기준 통계량 (판매속도지수_등급코드, 빈도_등급코드)
  방법: K-means 클러스터링 기반 (분포의 자연스러운 경계 찾기)
✓ Train set 등급코드 기준 판매속도지수 구간화 완료
  컬럼명: 판매속도지수_등급코드_구간
                count    pct
판매속도지수_등급코드_구간              
구간1             64357  43.42
구간2             70550  47.59
구간3             12425   8.38
구간4               800   0.54
구간5                99   0.07

✓ Train set 등급코드 기준 빈도 구간화 완료
  컬럼명: 빈도_등급코드_구간
            count    pct
빈도_등급코드_구간              
구간1         73267  50.25
구간2         40871  28.03
구간3         21979  15.07
구간4          7353   5.04
구간5          2333   1.60

✓ Test set 구간화 완료 (Train set 구간 경계 사용)
✓ 통계량 구간화 완료


In [25]:
# ============================================================
# 10. KModes 클러스터링 (Train set으로 학습)
# ============================================================
# KModes는 범주형 데이터를 사용하는 클러스터링 알고리즘
# - Train set으로 모델 학습
# - Test set에는 학습된 모델로 클러스터 라벨 할당
# - 등급코드 기준 통계량(구간화)을 포함하여 세분화된 특성 반영
# - 등급코드(3,697개) 자체는 제외하고, 구간화된 통계량만 사용
# - 모델코드 기준 통계량은 제외 (대표차종명이 이미 모델코드의 상위 개념을 포함)
# ============================================================

from kmodes.kmodes import KModes

print("\nKModes 클러스터링 중...")

# 클러스터링에 사용할 컬럼 선택
# - 등급코드(3,697개), 모델코드(706개), 세부등급코드(2,107개)는 너무 많 아서 제외
# - 구간화된 등급코드 기준 판매속도지수와 빈도 포함
#   → 등급코드 자체는 제외하지만, 등급코드 기준 통계량은 사용하여 세분화된 특성 반영
#   → 모델코드 기준 통계량은 제외 (대표차종명이 이미 모델코드의 상위 개념을 포함)
cluster_cols = ['국산외제차', '제조사코드', '대표차종명', '연료', '차종', 
                '배기량_구간',  # 국산차/외제차 구분
                '신차가격_구간',
                '판매속도지수_등급코드_구간',  # 등급코드 기준 판매속도지수 (구간화)
                '빈도_등급코드_구간']  # 등급코드 기준 빈도 (구간화)

# Train set에서 클러스터링 데이터 준비
X_train = df_train[cluster_cols].copy()

# 모든 값을 문자열로 변환
for col in X_train.columns:
    X_train[col] = X_train[col].astype(str)

X_train_array = X_train.values

# KModes 클러스터링 (Train set으로 학습)
best_seed = 2
n_clusters = 6
km = KModes(n_clusters=n_clusters, init="Huang", n_init=10, verbose=0, random_state=best_seed)

# Train set으로 학습 및 라벨 할당
train_cluster_labels = km.fit_predict(X_train_array)

# Train set에 클러스터 라벨 추가
df_train['cluster'] = train_cluster_labels

# 각 클러스터의 데이터 비율 확인
train_cluster_counts = pd.Series(train_cluster_labels).value_counts().sort_index()
print(f"\nTrain set 클러스터링 결과 (k={n_clusters}, random_state={best_seed}):")
for cluster_num, count in train_cluster_counts.items():
    pct = (count / len(train_cluster_labels)) * 100
    print(f"  Cluster {cluster_num}: {count:,}건 ({pct:.2f}%)")

print("✓ Train set 클러스터링 완료")

# ============================================================
# 10-1. KModes 모델 및 메타데이터 저장 (fit_predict 직후)
# ============================================================
# 학습된 모델을 저장하여 나중에 predict()로 사용할 수 있도록 함
# ============================================================

import pickle
import os
from datetime import datetime

# 결과 저장 폴더 생성
results_dir = '../../../00_data/04_results/kmodes_model'
os.makedirs(results_dir, exist_ok=True)

print("\nKModes 모델 및 메타데이터 저장 중...")

# 저장할 정보들
kmodes_model_info = {
    'model': km,  # KModes 모델 (fit_predict 후 학습된 모델)
    'cluster_cols': cluster_cols,  # 클러스터링에 사용한 컬럼 목록
    # 모델 생성 파라미터 (나중에 동일한 모델을 재생성할 수 있도록)
    'n_clusters': n_clusters,  # 클러스터 수
    'init': 'Huang',  # 초기화 방법
    'n_init': 10,  # 초기화 횟수
    'verbose': 0,  # 출력 레벨
    'random_state': best_seed,  # 랜덤 시드
    # 전처리 관련 정보
    '판매속도지수_등급코드_bins': 판매속도지수_등급코드_bins,  # 판매속도지수 구간 경계
    '빈도_등급코드_bins': 빈도_등급코드_bins,  # 빈도 구간 경계
    '판매속도지수_등급코드_stats': 판매속도지수_등급코드_stats,  # 판매속도지수 통계량 매핑
    '빈도_등급코드_stats': 빈도_등급코드_stats,  # 빈도 통계량 매핑
    # 메타데이터
    'saved_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),  # 저장 일시
    'train_size': len(df_train),  # Train set 크기
    'test_size': len(df_test),  # Test set 크기
}

# 모델 및 메타데이터 저장
model_file = os.path.join(results_dir, 'kmodes_model.pkl')
with open(model_file, 'wb') as f:
    pickle.dump(kmodes_model_info, f)

print(f"✓ 모델 저장 완료: {model_file}")

# 저장된 정보 요약 출력
print("\n저장된 정보:")
print(f"  - 모델: KModes (fit_predict 완료된 학습된 모델)")
print(f"  - 하이퍼파라미터:")
print(f"    * n_clusters: {n_clusters}")
print(f"    * init: Huang")
print(f"    * n_init: 10")
print(f"    * random_state: {best_seed}")
print(f"  - 클러스터링 컬럼 수: {len(cluster_cols)}개")
print(f"  - Train set 크기: {len(df_train):,}건")
print(f"  - Test set 크기: {len(df_test):,}건")
print(f"  - 저장 일시: {kmodes_model_info['saved_date']}")

print("\n✓ KModes 모델 및 메타데이터 저장 완료")



KModes 클러스터링 중...



Train set 클러스터링 결과 (k=6, random_state=2):
  Cluster 0: 36,943건 (24.92%)
  Cluster 1: 17,865건 (12.05%)
  Cluster 2: 31,624건 (21.33%)
  Cluster 3: 29,358건 (19.81%)
  Cluster 4: 9,917건 (6.69%)
  Cluster 5: 22,528건 (15.20%)
✓ Train set 클러스터링 완료

KModes 모델 및 메타데이터 저장 중...
✓ 모델 저장 완료: ../../../00_data/04_results/kmodes_model/kmodes_model.pkl

저장된 정보:
  - 모델: KModes (fit_predict 완료된 학습된 모델)
  - 하이퍼파라미터:
    * n_clusters: 6
    * init: Huang
    * n_init: 10
    * random_state: 2
  - 클러스터링 컬럼 수: 9개
  - Train set 크기: 148,235건
  - Test set 크기: 11,440건
  - 저장 일시: 2026-01-14 18:03:35

✓ KModes 모델 및 메타데이터 저장 완료


In [26]:
# ============================================================
# 9-2. 구간 피처 NaN 행 제거 (unseen 등급코드 처리)
# ============================================================
# prepare_for_seg 모듈에서 함수 가져오기 (이미 첫 번째 셀에서 로드됨)
# 숫자로 시작하는 모듈명이므로 importlib 사용
import importlib
prepare_for_seg = importlib.import_module('01_prepare_for_seg')

# 함수를 변수에 할당
remove_nan_rows_in_binned_features = prepare_for_seg.remove_nan_rows_in_binned_features

print("\n" + "="*70)
print("구간 피처 NaN 행 제거 (unseen 등급코드 처리)")
print("="*70)

# Train set NaN 제거
print("\n[Train set]")
before_train = len(df_train)
df_train = remove_nan_rows_in_binned_features(
    df_train,
    check_cols=['판매속도지수_등급코드_구간', '빈도_등급코드_구간']
)
after_train = len(df_train)

if before_train != after_train:
    print(f"  제거됨: {before_train - after_train}건")
    print(f"  남은 데이터: {after_train:,}건")

# Test set NaN 제거
print("\n[Test set]")
before_test = len(df_test)
df_test = remove_nan_rows_in_binned_features(
    df_test,
    check_cols=['판매속도지수_등급코드_구간', '빈도_등급코드_구간']
)
after_test = len(df_test)

if before_test != after_test:
    print(f"  제거됨: {before_test - after_test}건")
    print(f"  남은 데이터: {after_test:,}건")

print("\n✓ NaN 행 제거 완료")
print("="*70)



구간 피처 NaN 행 제거 (unseen 등급코드 처리)

[Train set]
  ⚠ 구간 피처 NaN 발견:
    - 판매속도지수_등급코드_구간: 4건
    - 빈도_등급코드_구간: 2432건
  → NaN 행 제거: 2436건 (1.64%)
  → 제거 후: 145,799건
  제거됨: 2436건
  남은 데이터: 145,799건

[Test set]
  ⚠ 구간 피처 NaN 발견:
    - 판매속도지수_등급코드_구간: 30건
    - 빈도_등급코드_구간: 237건
  → NaN 행 제거: 237건 (2.07%)
  → 제거 후: 11,203건
  제거됨: 237건
  남은 데이터: 11,203건

✓ NaN 행 제거 완료


In [27]:
# ============================================================
# 11. Test set에 클러스터 라벨 할당 (저장된 모델 불러와서 사용)
# ============================================================
# 저장된 모델을 불러와서 Test set에 예측하는 예제
# 실제 운영 환경에서는 이 방식으로 사용하면 됩니다
# ============================================================

import pickle
import os

print("\n저장된 KModes 모델 불러오기 중...")

# 저장된 모델 불러오기
results_dir = '../../../00_data/04_results/kmodes_model'
model_file = os.path.join(results_dir, 'kmodes_model.pkl')

if not os.path.exists(model_file):
    raise FileNotFoundError(f"모델 파일을 찾을 수 없습니다: {model_file}\n먼저 섹션 10-1에서 모델을 저장해주세요.")

with open(model_file, 'rb') as f:
    kmodes_model_info = pickle.load(f)

# 저장된 모델 및 메타데이터 추출
km_loaded = kmodes_model_info['model']
cluster_cols_loaded = kmodes_model_info['cluster_cols']

print("✓ 모델 불러오기 완료")
print(f"  - 저장 일시: {kmodes_model_info['saved_date']}")
print(f"  - 클러스터 수: {kmodes_model_info['n_clusters']}")
print(f"  - 클러스터링 컬럼 ({len(cluster_cols_loaded)}개): {cluster_cols_loaded}")

# Test set에 클러스터 라벨 할당
print("\nTest set에 클러스터 라벨 할당 중...")

# Test set에서 클러스터링 데이터 준비
X_test = df_test[cluster_cols_loaded].copy()

# 모든 값을 문자열로 변환
for col in X_test.columns:
    X_test[col] = X_test[col].astype(str)

X_test_array = X_test.values

# 저장된 모델로 Test set 예측
test_cluster_labels = km_loaded.predict(X_test_array)

# Test set에 클러스터 라벨 추가
df_test['cluster'] = test_cluster_labels

# 각 클러스터의 데이터 비율 확인
test_cluster_counts = pd.Series(test_cluster_labels).value_counts().sort_index()
print(f"\nTest set 클러스터 라벨 할당 결과:")
for cluster_num, count in test_cluster_counts.items():
    pct = (count / len(test_cluster_labels)) * 100
    print(f"  Cluster {cluster_num}: {count:,}건 ({pct:.2f}%)")

print("✓ Test set 클러스터 라벨 할당 완료")



저장된 KModes 모델 불러오기 중...
✓ 모델 불러오기 완료
  - 저장 일시: 2026-01-14 18:03:35
  - 클러스터 수: 6
  - 클러스터링 컬럼 (9개): ['국산외제차', '제조사코드', '대표차종명', '연료', '차종', '배기량_구간', '신차가격_구간', '판매속도지수_등급코드_구간', '빈도_등급코드_구간']

Test set에 클러스터 라벨 할당 중...

Test set 클러스터 라벨 할당 결과:
  Cluster 0: 2,866건 (25.58%)
  Cluster 1: 1,389건 (12.40%)
  Cluster 2: 2,493건 (22.25%)
  Cluster 3: 2,063건 (18.41%)
  Cluster 4: 672건 (6.00%)
  Cluster 5: 1,720건 (15.35%)
✓ Test set 클러스터 라벨 할당 완료


In [28]:
# ============================================================
# 12. Train/Test 데이터 저장 (클러스터 라벨 포함)
# ============================================================
# 클러스터 라벨이 할당된 Train/Test 데이터를 저장
# ============================================================

import os

print("\nTrain/Test 데이터 저장 중...")

# 결과 저장 폴더 (모델 저장 폴더와 동일)
results_dir = '03_배포용/00_data/03_result/kmodes_model'
os.makedirs(results_dir, exist_ok=True)

# Train set 저장
train_file = os.path.join(results_dir, 'df_train_cluster.csv')
df_train.to_csv(train_file, index=False, encoding='utf-8-sig')
print(f"✓ Train set 저장 완료: {train_file}")
print(f"  - 행 수: {len(df_train):,}건")
print(f"  - 컬럼 수: {len(df_train.columns)}개")
print(f"  - 클러스터 수: {df_train['cluster'].nunique()}개")

# Test set 저장
test_file = os.path.join(results_dir, 'df_test_cluster.csv')
df_test.to_csv(test_file, index=False, encoding='utf-8-sig')
print(f"\n✓ Test set 저장 완료: {test_file}")
print(f"  - 행 수: {len(df_test):,}건")
print(f"  - 컬럼 수: {len(df_test.columns)}개")
print(f"  - 클러스터 수: {df_test['cluster'].nunique()}개")

# 클러스터별 데이터 분포 요약
print("\n" + "="*70)
print("클러스터별 데이터 분포 요약")
print("="*70)

print("\nTrain set 클러스터 분포:")
train_cluster_summary = df_train['cluster'].value_counts().sort_index()
for cluster_num, count in train_cluster_summary.items():
    pct = (count / len(df_train)) * 100
    print(f"  Cluster {cluster_num}: {count:,}건 ({pct:.2f}%)")

print("\nTest set 클러스터 분포:")
test_cluster_summary = df_test['cluster'].value_counts().sort_index()
for cluster_num, count in test_cluster_summary.items():
    pct = (count / len(df_test)) * 100
    print(f"  Cluster {cluster_num}: {count:,}건 ({pct:.2f}%)")

print("\n✓ Train/Test 데이터 저장 완료")
print("="*70)



Train/Test 데이터 저장 중...


✓ Train set 저장 완료: 03_배포용/00_data/03_result/kmodes_model/df_train_cluster.csv
  - 행 수: 145,799건
  - 컬럼 수: 48개
  - 클러스터 수: 6개

✓ Test set 저장 완료: 03_배포용/00_data/03_result/kmodes_model/df_test_cluster.csv
  - 행 수: 11,203건
  - 컬럼 수: 48개
  - 클러스터 수: 6개

클러스터별 데이터 분포 요약

Train set 클러스터 분포:
  Cluster 0: 34,511건 (23.67%)
  Cluster 1: 17,865건 (12.25%)
  Cluster 2: 31,621건 (21.69%)
  Cluster 3: 29,358건 (20.14%)
  Cluster 4: 9,917건 (6.80%)
  Cluster 5: 22,527건 (15.45%)

Test set 클러스터 분포:
  Cluster 0: 2,866건 (25.58%)
  Cluster 1: 1,389건 (12.40%)
  Cluster 2: 2,493건 (22.25%)
  Cluster 3: 2,063건 (18.41%)
  Cluster 4: 672건 (6.00%)
  Cluster 5: 1,720건 (15.35%)

✓ Train/Test 데이터 저장 완료
